# TPI – Introducción al Análisis de Datos (División 343)
**Caso de estudio:** Telco Customer Churn

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rociojuarez/analisis-de-datos-telco-churn/blob/main/notebooks/TPI_Analisis_de_Datos.ipynb)


---
# PARTE 1 – Fundamentos, ingesta y limpieza de datos

## Paso 1 · Problema de negocio, hipótesis y variable objetivo

**¿Qué es esto?** Antes de tocar datos hay que decir *para qué* los vamos a analizar.

- **Problema de negocio:** una empresa de telecomunicaciones pierde clientes que cancelan su servicio (*churn*). Retener a un cliente cuesta menos que conseguir uno nuevo, así que queremos entender **qué características tienen los clientes que se van** para poder actuar antes.
- **Variable objetivo (target):** `Churn` (Yes/No). Es lo que queremos explicar o predecir.
- **Hipótesis analíticas:** afirmaciones que vamos a verificar con los datos más adelante (en el EDA):
  1. Los clientes con contrato *Month-to-month* cancelan más que los de contratos largos.
  2. Los clientes con poca antigüedad (`tenure` baja) cancelan más.
  3. Cargos mensuales altos (`MonthlyCharges`) se asocian a más cancelación.
  4. Pagar con *Electronic check* se asocia a más cancelación.


## Paso 2 · Cargar los datos y primer vistazo

**¿Qué hacemos?** Leemos el CSV con `pandas` en un *DataFrame* (una tabla en memoria) y miramos su tamaño, sus primeras filas y el tipo de dato (`dtype`) que pandas le asignó a cada columna.

Para que el notebook funcione **tanto en Google Colab como en local**, intentamos leer el CSV directo desde el repositorio de GitHub; si eso falla (sin internet o repo no disponible), usamos la ruta local `../data/raw/`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Dónde está el dato original en el repositorio de GitHub
GITHUB_USER = "rociojuarez"
GITHUB_REPO = "analisis-de-datos-telco-churn"
GITHUB_BRANCH = "main"
NOMBRE_CSV = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
URL_RAW = (f"https://raw.githubusercontent.com/{GITHUB_USER}/{GITHUB_REPO}/"
           f"{GITHUB_BRANCH}/data/raw/{NOMBRE_CSV}")

# Plan B: ruta local (cuando se corre desde la carpeta notebooks/ del repo clonado)
RUTA_RAW = Path("../data/raw") / NOMBRE_CSV

try:
    df = pd.read_csv(URL_RAW)                       # Colab: lee el CSV directo desde GitHub
    print("Datos cargados desde GitHub.")
except Exception as error:
    print(f"No se pudo leer desde GitHub ({type(error).__name__}). Se usa la ruta local.")
    df = pd.read_csv(RUTA_RAW)                      # Local: lee el archivo de la carpeta data/raw

print("Filas, columnas:", df.shape)
df.head()

In [2]:
# Tipo de dato que pandas asignó por defecto a cada columna
df.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

### Lectura rápida de lo que acabamos de ver
- Hay **7043 filas** (clientes) y **21 columnas** (variables).
- `TotalCharges` aparece como `object` (texto), aunque debería ser un número. **Esa es la primera trampa del dataset** y la resolvemos en el Paso 4.
- `SeniorCitizen` aparece como `int64` (0/1), pero en realidad es una categoría (Sí/No).

## Paso 3 · Clasificar el dataset y las variables

**Tipo de dato según su estructura:** el dataset es **estructurado**: está organizado en filas (clientes) y columnas (variables) con un formato fijo, como una tabla de base de datos. (Semi-estructurado sería algo como JSON; no estructurado, texto libre, imágenes o audio.)

| Variable | Tipo teórico | `dtype` de pandas | ¿Coincide? |
|---|---|---|---|
| `customerID` | Cualitativa nominal (identificador, no sirve para analizar) | object | ✅ |
| `gender`, `Partner`, `Dependents`, `PhoneService`, `PaperlessBilling` | Cualitativa nominal (binaria) | object | ✅ |
| `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`, `PaymentMethod` | Cualitativa nominal | object | ✅ |
| `SeniorCitizen` | Cualitativa nominal (binaria) | **int64** | ❌ es categoría guardada como número |
| `Contract` | Cualitativa **ordinal** (Month-to-month < One year < Two year) | object | ✅ (pandas no sabe que hay orden) |
| `tenure` | Cuantitativa **discreta** (meses de antigüedad) | int64 | ✅ |
| `MonthlyCharges` | Cuantitativa **continua** | float64 | ✅ |
| `TotalCharges` | Cuantitativa **continua** | **object** | ❌ es número guardado como texto |
| `Churn` | Cualitativa nominal (binaria) – **variable objetivo** | object | ✅ |

---
## Paso 4 · Limpieza: `TotalCharges` y valores faltantes

**El problema:** `TotalCharges` es un monto (cuantitativa continua) pero pandas la leyó como texto. La causa: algunas filas tienen un **espacio en blanco `" "`** en lugar de un número. Pandas no lo considera un nulo, lo trata como un texto más, y eso contamina toda la columna.

**Qué vamos a hacer:**
1. Trabajar sobre una **copia** del DataFrame (`df_clean`), para no modificar `df`.
2. Detectar las filas con `TotalCharges` vacío y ver qué tienen en común.
3. Convertir la columna a número (los vacíos pasan a ser nulos reales, `NaN`).
4. Tratar esos nulos.

In [ ]:
# Trabajamos sobre una COPIA: df queda intacto como referencia
df_clean = df.copy()

# Filas donde TotalCharges es solo espacios en blanco
vacios = df_clean["TotalCharges"].str.strip() == ""
print("Filas con TotalCharges vacío:", vacios.sum())
print("Valores de tenure en esas filas:", df_clean.loc[vacios, "tenure"].unique())

df_clean.loc[vacios, ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]]

**Lectura:** hay **11 filas** con `TotalCharges` vacío y **todas tienen `tenure = 0`**. Son clientes recién incorporados que todavía no completaron ningún ciclo de facturación, así que aún no tienen un total acumulado.

**Decisión de tratamiento: imputar con 0.**
- *Justificación:* el vacío no es un dato perdido al azar; refleja que el cliente todavía no pagó nada. El valor 0 representa fielmente esa situación (`tenure = 0` ⇒ total acumulado = 0).
- *Por qué no eliminar:* son solo 11 filas (≈0,16 %), pero son clientes reales y eliminarlos sería perder información sin necesidad.
- *Por qué no usar la mediana:* les asignaría pagos que nunca hicieron.
- *Limitación a reconocer:* un 0 podría confundirse con un cliente que sí pagó y suma cero; acá se descarta porque todos tienen `tenure = 0`.

In [ ]:
# Convertir a número: los espacios en blanco pasan a NaN (nulo real)
df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce")
print("Nulos tras convertir:", df_clean["TotalCharges"].isna().sum())

# Imputar con 0 (clientes nuevos sin pagos acumulados)
df_clean["TotalCharges"] = df_clean["TotalCharges"].fillna(0)

print("Nulos tras imputar:", df_clean["TotalCharges"].isna().sum())
print("dtype de TotalCharges:", df_clean["TotalCharges"].dtype)
print("Nulos en todo el dataset:", df_clean.isna().sum().sum())

**Resultado esperado:** 11 nulos tras convertir, 0 tras imputar, `TotalCharges` como `float64` y ningún nulo en el dataset. Con esto `TotalCharges` ya se puede usar para estadística, gráficos y modelos.

---
## Paso 5 · Adecuación de tipos de datos (`dtypes`)

**Objetivo:** que el tipo de dato de cada columna en pandas refleje su **tipo real** (el que definimos en el Paso 3).

**Qué corregimos**
- `SeniorCitizen`: es una categoría (sí/no) guardada como número 0/1. La pasamos a `No`/`Yes`, igual que las demás binarias, para que sea coherente.
- Variables **nominales** → tipo `category` (categorías sin orden).
- `Contract` → `category` **ordenada** (Month-to-month < One year < Two year), porque es ordinal.
- `TotalCharges` ya quedó en `float64` en el Paso 4; `tenure` (`int64`) y `MonthlyCharges` (`float64`) ya estaban bien.
- `customerID` queda como texto: es un identificador, no una categoría para analizar (tiene un valor distinto por cliente).

In [ ]:
# 5.1 SeniorCitizen: de 0/1 a No/Yes, igual que el resto de las columnas binarias
df_clean["SeniorCitizen"] = df_clean["SeniorCitizen"].map({0: "No", 1: "Yes"})
print("Nulos tras el mapeo (debe ser 0):", df_clean["SeniorCitizen"].isna().sum())

memoria_antes = df_clean.memory_usage(deep=True).sum() / 1024**2

# 5.2 Variables cualitativas NOMINALES -> tipo category
cols_nominales = [
    "gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService",
    "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
    "PaperlessBilling", "PaymentMethod", "Churn",
]
df_clean[cols_nominales] = df_clean[cols_nominales].astype("category")

# 5.3 Contract es ORDINAL: category con orden explícito
orden_contrato = ["Month-to-month", "One year", "Two year"]
df_clean["Contract"] = pd.Categorical(
    df_clean["Contract"], categories=orden_contrato, ordered=True
)

memoria_despues = df_clean.memory_usage(deep=True).sum() / 1024**2
print(f"Memoria antes: {memoria_antes:.2f} MB | después: {memoria_despues:.2f} MB")
print("Contract ordenado:", df_clean["Contract"].cat.ordered, list(df_clean["Contract"].cat.categories))
df_clean.dtypes

**Resultado esperado:** las columnas cualitativas pasan a `category` (con `Contract` ordenada), las numéricas conservan `int64`/`float64` y `customerID` sigue como texto. El uso de memoria baja considerablemente, porque `category` guarda cada etiqueta una sola vez.

---
## Paso 6 · Chequeo de calidad de datos
**Objetivo:** antes de analizar, comprobar que los datos son confiables. Revisamos cuatro cosas:
1. **Duplicados** (filas o clientes repetidos).
2. **Categorías** raras o mal escritas.
3. **Consistencia lógica** entre columnas relacionadas (ej.: sin internet no puede haber servicios de streaming contratados).
4. **Rangos** de las variables numéricas (valores imposibles).

In [ ]:
# 6.1 Duplicados: filas completas repetidas y customerID repetidos
print("Filas duplicadas:", df_clean.duplicated().sum())
print("customerID duplicados:", df_clean["customerID"].duplicated().sum())
print("Nulos en todo el dataset:", df_clean.isna().sum().sum())

In [ ]:
# 6.2 Categorías: ¿hay valores raros, mal escritos o con espacios de más?
cols_categoricas = df_clean.select_dtypes(include="category").columns
for col in cols_categoricas:
    print(f"{col:18s}", list(df_clean[col].cat.categories))

**Lectura:** las categorías están bien escritas y son consistentes. Ojo: *"No internet service"* y *"No phone service"* **no son errores**: significan "no aplica" (el servicio depende de tener internet o telefonía).

In [ ]:
# 6.3 Consistencia lógica entre columnas
servicios_internet = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                      "TechSupport", "StreamingTV", "StreamingMovies"]
sin_internet = df_clean["InternetService"] == "No"

# Sin internet => los servicios adicionales deben decir "No internet service" (y solo ahí)
for col in servicios_internet:
    ok = ((df_clean[col] == "No internet service") == sin_internet).all()
    print(f"{col:18s} consistente con InternetService:", ok)

# Sin telefonía => MultipleLines debe decir "No phone service" (y solo ahí)
sin_telefono = df_clean["PhoneService"] == "No"
print("MultipleLines consistente con PhoneService:",
      ((df_clean["MultipleLines"] == "No phone service") == sin_telefono).all())

In [ ]:
# 6.4 Rangos de las variables numéricas: ¿hay valores imposibles (negativos, etc.)?
df_clean[["tenure", "MonthlyCharges", "TotalCharges"]].describe().round(2)

**Conclusión del chequeo:** sin duplicados, sin nulos, categorías consistentes, relaciones lógicas entre columnas respetadas y rangos plausibles (`tenure` entre 0 y 72 meses, cargos positivos). El dataset está en condiciones de pasar al análisis de outliers.

---
## Paso 7 · Detección y gestión de valores atípicos (outliers)
**¿Qué es un outlier?** Un valor muy alejado del resto, que puede ser un error de carga o un caso real pero extremo.

**Método del rango intercuartílico (IQR):**
- `Q1` = percentil 25, `Q3` = percentil 75, `IQR = Q3 − Q1` (el rango donde está el 50 % central de los datos).
- Se considera outlier todo valor **menor a `Q1 − 1,5·IQR`** o **mayor a `Q3 + 1,5·IQR`**.
- El **boxplot** (diagrama de caja) dibuja esa misma regla: la caja va de Q1 a Q3 y los puntos fuera de los bigotes son los outliers.

Lo aplicamos a las tres variables numéricas: `tenure`, `MonthlyCharges` y `TotalCharges`.

In [ ]:
# 7.1 Método IQR: límites a 1.5 * IQR por debajo de Q1 y por encima de Q3
cols_numericas = ["tenure", "MonthlyCharges", "TotalCharges"]

def limites_iqr(serie):
    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

resumen = []
for col in cols_numericas:
    inf, sup = limites_iqr(df_clean[col])
    n_out = ((df_clean[col] < inf) | (df_clean[col] > sup)).sum()
    resumen.append({"variable": col, "límite inferior": round(inf, 2),
                    "límite superior": round(sup, 2), "mínimo": df_clean[col].min(),
                    "máximo": df_clean[col].max(), "outliers": n_out})

pd.DataFrame(resumen)

In [ ]:
# 7.2 Boxplots: los puntos fuera de los "bigotes" serían outliers
import matplotlib.pyplot as plt
import seaborn as sns

fig, ejes = plt.subplots(1, 3, figsize=(13, 4))
for eje, col in zip(ejes, cols_numericas):
    sns.boxplot(y=df_clean[col], ax=eje, color="skyblue")
    eje.set_title(col)
plt.tight_layout()
plt.show()

**Lectura y decisión:** con el criterio IQR **no se detecta ningún outlier** en las tres variables: todos los mínimos y máximos caen dentro de los límites, y los boxplots no muestran puntos fuera de los bigotes.

**Decisión: no hay nada que tratar, se conservan todos los valores.** Además, aunque hubiera aparecido alguno, serían valores plausibles del negocio (por ejemplo, clientes de mucha antigüedad con facturación total alta), no errores; eliminarlos habría quitado información real.

**Con esto cerramos la limpieza del dataset.** Falta crear variables calculadas, eliminar atributos irrelevantes y exportar el dataset depurado a `data/processed`.

---
# PARTE 2 – Manipulación de datos con Pandas

## Paso 8 · Filtrado avanzado y agregación (`groupby`)

**Filtrado con múltiples condiciones:** seleccionamos subconjuntos de clientes combinando condiciones con `&` (y) y `|` (o).

**Agregación:** `groupby` agrupa los clientes por una categoría y calcula varias métricas a la vez (conteo, promedio, suma), que es una forma de **segmentar el negocio**.

In [ ]:
# 8.1 Filtrado con múltiples condiciones
# &  = "Y": deben cumplirse todas | = "O": alcanza con que se cumpla una
# Cada condición va ENTRE PARÉNTESIS.
churn_global = (df_clean["Churn"] == "Yes").mean() * 100
print(f"Tasa de churn global: {churn_global:.1f}%")

# Filtro 1 (&): clientes nuevos, con contrato mensual y cargo mensual alto
f1 = ((df_clean["Contract"] == "Month-to-month")
      & (df_clean["MonthlyCharges"] > 70)
      & (df_clean["tenure"] <= 12))
nuevos_caros = df_clean[f1]
print(f"Nuevos + mensual + caro: {len(nuevos_caros)} clientes | "
      f"churn: {(nuevos_caros['Churn'] == 'Yes').mean() * 100:.1f}%")

# Filtro 2 (|): clientes que pagan con cheque (electrónico o por correo)
f2 = ((df_clean["PaymentMethod"] == "Electronic check")
      | (df_clean["PaymentMethod"] == "Mailed check"))
pago_cheque = df_clean[f2]
print(f"Pagan con cheque: {len(pago_cheque)} clientes | "
      f"churn: {(pago_cheque['Churn'] == 'Yes').mean() * 100:.1f}%")

nuevos_caros[["customerID", "tenure", "MonthlyCharges", "Contract", "Churn"]].head()

**Lectura:** la tasa de churn global es ≈26,5 %. El grupo de clientes **nuevos, con contrato mensual y cargo alto** (856 clientes) cancela muchísimo más (≈69 %). Quienes pagan con cheque (3977 clientes) cancelan ≈34,7 %, por encima del promedio.

In [ ]:
# 8.2 Agregación con groupby: varias métricas a la vez
def pct_churn(serie):
    """Porcentaje de clientes que cancelaron en cada grupo."""
    return (serie == "Yes").mean() * 100

metricas = dict(
    clientes=("customerID", "count"),
    churn_pct=("Churn", pct_churn),
    cargo_mensual_prom=("MonthlyCharges", "mean"),
    antiguedad_prom=("tenure", "mean"),
    facturacion_total=("TotalCharges", "sum"),
)

por_contrato = df_clean.groupby("Contract", observed=True).agg(**metricas).round(1)
por_contrato

**Lectura:** el churn baja drásticamente con la duración del contrato: ≈42,7 % en *Month-to-month*, ≈11,3 % en *One year* y ≈2,8 % en *Two year*. También crece la antigüedad promedio (18 → 42 → 57 meses).

In [ ]:
# 8.3 Mismo resumen por método de pago, ordenado por churn
por_pago = (df_clean.groupby("PaymentMethod", observed=True)
            .agg(**metricas).round(1)
            .sort_values("churn_pct", ascending=False))
por_pago

**Lectura:** *Electronic check* tiene el churn más alto (≈45,3 %), muy por encima del resto (≈15–19 %).

> ⚠️ Estas son **observaciones preliminares** que apuntan en la dirección de las hipótesis 1 y 4 del Paso 1. Que dos cosas vayan juntas no prueba que una cause la otra (por ejemplo, los contratos mensuales también tienen clientes más nuevos). La comprobación formal la hacemos en el análisis exploratorio.

In [ ]:
# 8.4 groupby por DOS variables a la vez: contrato y tipo de internet
por_contrato_internet = (df_clean.groupby(["Contract", "InternetService"], observed=True)
                         .agg(clientes=("customerID", "count"),
                              churn_pct=("Churn", pct_churn),
                              cargo_mensual_prom=("MonthlyCharges", "mean"))
                         .round(1))
por_contrato_internet

**Lectura:** al cruzar dos variables se ve algo que el resumen por contrato solo ocultaba: **dentro de cada tipo de contrato, los clientes con fibra óptica cancelan mucho más** que los de DSL o sin internet (por ejemplo, en contrato mensual: ≈54,6 % con fibra contra ≈32,2 % con DSL y ≈18,9 % sin internet). La peor combinación es *Month-to-month + Fiber optic* (2128 clientes).

> ⚠️ Es una observación preliminar: la fibra también es el servicio más caro (cargo mensual promedio ≈87 contra ≈50 en DSL), así que no se puede atribuir la cancelación solo al tipo de internet. Se profundiza en el análisis exploratorio.

---
## Paso 9 · Transformación básica: renombrar, eliminar y crear columnas

Tres operaciones que preparan el dataset para el análisis:
1. **Renombrar** las columnas a español y con un formato uniforme (minúsculas con guion bajo).
2. **Eliminar atributos irrelevantes:** `customerID` solo identifica al cliente, no aporta información para explicar la cancelación.
3. **Crear columnas calculadas** con operaciones aritméticas entre variables existentes.

> ⚠️ **A partir de acá las columnas tienen los nombres nuevos** (ver el diccionario de la celda 9.1). Los *valores* de las categorías (Yes/No, Month-to-month, etc.) se mantienen en inglés.

In [ ]:
# 9.1 Renombrar columnas a español (formato minúsculas_con_guion_bajo)
mapa_columnas = {
    "customerID": "id_cliente",
    "gender": "genero",
    "SeniorCitizen": "adulto_mayor",
    "Partner": "tiene_pareja",
    "Dependents": "tiene_dependientes",
    "tenure": "antiguedad_meses",
    "PhoneService": "servicio_telefonico",
    "MultipleLines": "lineas_multiples",
    "InternetService": "servicio_internet",
    "OnlineSecurity": "seguridad_online",
    "OnlineBackup": "respaldo_online",
    "DeviceProtection": "proteccion_dispositivo",
    "TechSupport": "soporte_tecnico",
    "StreamingTV": "streaming_tv",
    "StreamingMovies": "streaming_peliculas",
    "Contract": "contrato",
    "PaperlessBilling": "factura_electronica",
    "PaymentMethod": "metodo_pago",
    "MonthlyCharges": "cargo_mensual",
    "TotalCharges": "cargo_total",
    "Churn": "churn",
}
df_clean = df_clean.rename(columns=mapa_columnas)
print(list(df_clean.columns))

In [ ]:
# 9.2 Eliminar atributos irrelevantes: id_cliente es solo un identificador
# (verificamos antes, en el Paso 6, que no hay IDs duplicados)
df_clean = df_clean.drop(columns=["id_cliente"])
print("Forma del dataset:", df_clean.shape)

**Columnas calculadas que vamos a crear**
| Columna | Fórmula | Para qué sirve |
|---|---|---|
| `gasto_mensual_real` | `cargo_total / antiguedad_meses` | Cuánto pagó en promedio por mes a lo largo de su historia (si la antigüedad es 0, se usa la cuota mensual vigente) |
| `diferencia_cargo` | `cargo_mensual − gasto_mensual_real` | Si hoy paga más o menos que su promedio histórico (cambios de plan o servicios) |
| `cant_servicios_extra` | cuenta de servicios adicionales con "Yes" | Qué tan "enganchado" está el cliente con la empresa |

In [ ]:
# 9.3 Columnas calculadas con operaciones aritméticas entre variables

# a) Gasto mensual promedio REAL: total pagado / meses de antigüedad.
#    Si antiguedad_meses = 0 no se puede dividir: usamos la cuota mensual vigente.
df_clean["gasto_mensual_real"] = np.where(
    df_clean["antiguedad_meses"] > 0,
    df_clean["cargo_total"] / df_clean["antiguedad_meses"],
    df_clean["cargo_mensual"],
).round(2)

# b) Diferencia entre la cuota actual y el gasto promedio histórico
df_clean["diferencia_cargo"] = (df_clean["cargo_mensual"] - df_clean["gasto_mensual_real"]).round(2)

# c) Cantidad de servicios adicionales contratados (de 0 a 6)
servicios = ["seguridad_online", "respaldo_online", "proteccion_dispositivo",
             "soporte_tecnico", "streaming_tv", "streaming_peliculas"]
df_clean["cant_servicios_extra"] = (df_clean[servicios] == "Yes").sum(axis=1)

print("Forma final:", df_clean.shape, "| nulos:", df_clean.isna().sum().sum())
df_clean[["antiguedad_meses", "cargo_mensual", "cargo_total",
          "gasto_mensual_real", "diferencia_cargo", "cant_servicios_extra"]].head()

---
## Paso 10 · Exportar el dataset depurado

Guardamos el resultado de toda la limpieza en `data/processed/telco_churn_limpio.csv`. Ese archivo es el **punto de partida del análisis exploratorio y del modelado**: quien trabaje en esas etapas no necesita repetir la limpieza.

El dato original en `data/raw` **no se modifica**: siempre se puede reproducir todo desde cero.

In [ ]:
# 10.1 Exportar el dataset depurado a data/processed
import os

# Si existe ../data (corriendo desde notebooks/) se guarda ahí; en Colab se crea data/processed en la sesión
CARPETA_PROCESSED = "../data/processed" if os.path.isdir("../data") else "data/processed"
RUTA_PROCESSED = f"{CARPETA_PROCESSED}/telco_churn_limpio.csv"
os.makedirs(CARPETA_PROCESSED, exist_ok=True)                 # crea la carpeta si no existe
df_clean.to_csv(RUTA_PROCESSED, index=False)                  # index=False: no guarda el índice 0..7042

# Verificación: leemos lo que acabamos de guardar
df_check = pd.read_csv(RUTA_PROCESSED)
print("Exportado:", df_check.shape, "| nulos:", df_check.isna().sum().sum())
print("Mismas columnas que df_clean:", list(df_check.columns) == list(df_clean.columns))

**Importante:** un CSV **no guarda los tipos de dato**. Al leerlo, las columnas categóricas vuelven a ser texto y se pierde el orden de `contrato`. La celda siguiente muestra cómo restaurarlos.

In [ ]:
# 10.2 Cómo cargar este dataset en los pasos siguientes (EDA y modelado)
# Un CSV no guarda los tipos: al leerlo, las categorías vuelven a ser texto. Se restauran así:
cols_numericas = ["antiguedad_meses", "cargo_mensual", "cargo_total",
                  "gasto_mensual_real", "diferencia_cargo", "cant_servicios_extra"]
cols_categoricas = [c for c in df_check.columns if c not in cols_numericas]

df_listo = pd.read_csv(RUTA_PROCESSED, dtype={c: "category" for c in cols_categoricas})
df_listo["contrato"] = pd.Categorical(
    df_listo["contrato"], categories=["Month-to-month", "One year", "Two year"], ordered=True
)
df_listo.dtypes.astype(str).value_counts()

**Cierre de la Parte 1 y 2:** el dataset quedó depurado (**7043 filas × 23 columnas**, sin nulos) y exportado. Siguen el análisis exploratorio y visualización, y el preprocesamiento y modelado.